# X-ray Tomography Reconstruction

Complete implementation of Tikhonov and Total Variation regularization methods.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path
sys.path.append('..')

Path('../results/figures').mkdir(parents=True, exist_ok=True)
Path('../results/reconstructions').mkdir(parents=True, exist_ok=True)

from src.data_utils import (
    load_density_example, load_toy_measurements, load_3d_data,
    measurements_to_vector, grid_to_vector, vector_to_grid, save_reconstruction
)
from src.matrix_construction import (
    build_toy_ray_path_matrix, build_derivative_matrix_2d, build_derivative_matrix_3d,
    build_combined_derivative_matrix, compute_gradient_magnitude
)
from src.solvers import cgls, irls, gradient_descent
from src.visualization import (
    visualize_2d_field, visualize_gradients, plot_convergence,
    visualize_3d_slices, compare_reconstructions, plot_histogram
)

%matplotlib inline

# Q3-Q4: Derivative Matrices and Gradients

## Load Example Data

In [ ]:
X1 = load_density_example('../data/X1.mat')
X2 = load_density_example('../data/X2.mat')
X3 = load_density_example('../data/X3.mat')

print(f"X1 shape: {X1.shape}")
print(f"X2 shape: {X2.shape}")
print(f"X3 shape: {X3.shape}")

## Compute Gradients for X1

In [ ]:
M, N = X1.shape
Dx = build_derivative_matrix_2d(M, N, 'x')
Dy = build_derivative_matrix_2d(M, N, 'y')

x = grid_to_vector(X1)
Dx_x = Dx @ x
Dy_x = Dy @ x
G1 = compute_gradient_magnitude(Dx_x, Dy_x, M, N)

visualize_gradients(X1, Dx_x, Dy_x, G1, title="X1 Gradient Analysis",
                   save_path='../results/figures/X1_gradients.png')

## Compute Gradients for X2

In [ ]:
M, N = X2.shape
Dx = build_derivative_matrix_2d(M, N, 'x')
Dy = build_derivative_matrix_2d(M, N, 'y')

x = grid_to_vector(X2)
Dx_x = Dx @ x
Dy_x = Dy @ x
G2 = compute_gradient_magnitude(Dx_x, Dy_x, M, N)

visualize_gradients(X2, Dx_x, Dy_x, G2, title="X2 Gradient Analysis",
                   save_path='../results/figures/X2_gradients.png')

## Compute Gradients for X3

In [ ]:
M, N = X3.shape
Dx = build_derivative_matrix_2d(M, N, 'x')
Dy = build_derivative_matrix_2d(M, N, 'y')

x = grid_to_vector(X3)
Dx_x = Dx @ x
Dy_x = Dy @ x
G3 = compute_gradient_magnitude(Dx_x, Dy_x, M, N)

visualize_gradients(X3, Dx_x, Dy_x, G3, title="X3 Gradient Analysis",
                   save_path='../results/figures/X3_gradients.png')

## Q4: Analysis

Gradient magnitude highlights edges where density changes rapidly. Horizontal derivatives detect vertical edges, vertical derivatives detect horizontal edges.

# Q8-Q9: Gradient Descent Analysis

## Setup Toy Problem

In [ ]:
Y = load_toy_measurements('../data/Y.mat')
y = measurements_to_vector(Y)

M, N = 5, 5
A = build_toy_ray_path_matrix(M, N, delta_x=1.0, delta_y=1.0)
L = build_combined_derivative_matrix(M, N, dimensions=2)

lam = 1e-5
print(f"A shape: {A.shape}")
print(f"L shape: {L.shape}")
print(f"λ = {lam}")

## Build Q Matrix and Compute Eigenvalues

In [ ]:
from scipy import sparse

A_sparse = sparse.csr_matrix(A)
Q = A_sparse.T @ A_sparse + lam * (L.T @ L)
Q = Q.toarray()
b = -A_sparse.T @ y

eigenvalues = np.linalg.eigvalsh(Q)
lambda_min = eigenvalues[0]
lambda_max = eigenvalues[-1]
kappa = lambda_max / lambda_min

print(f"λ_min = {lambda_min:.6e}")
print(f"λ_max = {lambda_max:.6e}")
print(f"Condition number κ(Q) = {kappa:.6e}")

## Step Size and Convergence Rate

In [ ]:
import math

alpha_max = 2.0 / lambda_max
alpha_optimal = 2.0 / (lambda_min + lambda_max)

print(f"Valid range: 0 < α < {alpha_max:.6e}")
print(f"Optimal α* = {alpha_optimal:.6e}")

rho = (kappa - 1) / (kappa + 1)
k_estimated = math.log(0.1) / (2 * math.log(rho))

print(f"\nConvergence rate ρ = {rho:.6f}")
print(f"Estimated iterations for 10x reduction: {k_estimated:.1f}")

## Test Gradient Descent vs Conjugate Gradient

In [ ]:
step_size = 1.5 / lambda_max

result_gd = gradient_descent(Q, b, step_size=step_size, max_iter=2000, tol=1e-8)
result_cg = cgls(A, y, L, lam=lam, tol=1e-8, max_iter=2000)

print(f"Gradient Descent: {result_gd.iterations} iterations")
print(f"Conjugate Gradient: {result_cg.iterations} iterations")

if result_cg.iterations > 0:
    print(f"Speedup: {result_gd.iterations / result_cg.iterations:.1f}x")
else:
    print("Note: Both converged immediately (toy matrix A not implemented)")

## Convergence Comparison

In [ ]:
if len(result_gd.residuals) > 1 or len(result_cg.residuals) > 1:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.semilogy(result_gd.residuals, 'b-', linewidth=2, label='Gradient Descent', alpha=0.7)
    ax1.semilogy(result_cg.residuals, 'r--', linewidth=2, label='Conjugate Gradient', alpha=0.7)
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('Residual Norm')
    ax1.set_title('Residual Convergence')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(result_gd.objective_values, 'b-', linewidth=2, label='Gradient Descent', alpha=0.7)
    ax2.plot(result_cg.objective_values, 'r--', linewidth=2, label='Conjugate Gradient', alpha=0.7)
    ax2.set_xlabel('Iteration')
    ax2.set_ylabel('Objective Value')
    ax2.set_title('Objective Convergence')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('../results/figures/gd_vs_cg.png', dpi=150)
    plt.show()
else:
    print("Skipping plot - both solvers converged in 0 iterations (implement build_toy_ray_path_matrix first)")

# Q10-Q11: CGLS and Small Bag Reconstruction

## Q10: Test CGLS on Toy Problem

In [ ]:
lambda_values = [1e-6, 1e-5, 1e-4, 1e-3]
results_toy = {}

for lam in lambda_values:
    result = cgls(A, y, L, lam=lam, tol=1e-6, max_iter=200)
    results_toy[lam] = result
    print(f"λ={lam:.0e}: converged={result.converged}, iter={result.iterations}, obj={result.objective_values[-1]:.6e}")

## Visualize Toy Reconstructions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
axes = axes.flatten()

for idx, lam in enumerate(lambda_values):
    X_recon = vector_to_grid(results_toy[lam].x, (M, N))
    im = axes[idx].imshow(X_recon, cmap='gray', origin='lower')
    axes[idx].set_title(f'λ = {lam:.0e}')
    plt.colorbar(im, ax=axes[idx])

plt.tight_layout()
plt.savefig('../results/figures/toy_reconstructions.png', dpi=150)
plt.show()

## Q11: Small Bag Reconstruction

In [ ]:
y_small, A_small = load_3d_data('../data/Small')
n = 19
L_3d = build_combined_derivative_matrix(n, n, dimensions=3)

## Test Different Lambda Values

In [ ]:
lambda_values_3d = [1e-6, 1e-5, 1e-4, 1e-3]
results_3d = {}

for lam in lambda_values_3d:
    print(f"\nSolving with λ = {lam}...")
    result = cgls(A_small, y_small, L_3d, lam=lam, tol=1e-6, max_iter=300)
    results_3d[lam] = result
    
    print(f"  Converged: {result.converged}, Iterations: {result.iterations}")
    
    X_recon = vector_to_grid(result.x, (n, n, n))
    visualize_3d_slices(X_recon, axis='z', num_slices=9,
                       title=f"Small Bag (λ={lam})",
                       save_path=f'../results/figures/small_lam{lam}.png')

## Choose Best Lambda

In [ ]:
best_lam = 1e-5
best_result = results_3d[best_lam]

plot_convergence(best_result.residuals, best_result.objective_values,
                title=f"CGLS Convergence (λ={best_lam})",
                save_path='../results/figures/small_convergence.png')

# Q12: Curve Fitting - L1 vs L2 Regularization

## Define Functions

In [ ]:
x = np.array([-4, -3, -2, -1, 0, 1, 2, 3, 4])
n_curve = len(x)

f1 = np.where(x < 0, 0.0, 1.0)
f1[4] = 0.5

f2 = 1.0 / (1.0 + np.exp(-2 * x))

print("Function 1 (Step):")
for xi, fi in zip(x, f1):
    print(f"  x={xi:2d}: f1={fi:.4f}")

print("\nFunction 2 (Smooth):")
for xi, fi in zip(x, f2):
    print(f"  x={xi:2d}: f2={fi:.4f}")

## Build 1D Derivative Matrix

In [ ]:
def build_derivative_matrix_1d(n):
    Dx = np.zeros((n, n))
    for i in range(n - 1):
        Dx[i, i] = -1.0
        Dx[i, i + 1] = 1.0
    return Dx

Dx = build_derivative_matrix_1d(n_curve)
Df1 = Dx @ f1
Df2 = Dx @ f2

## Compute Norms

In [ ]:
l1_f1 = np.sum(np.abs(Df1))
l1_f2 = np.sum(np.abs(Df2))
l2_f1 = np.sqrt(np.sum(Df1**2))
l2_f2 = np.sqrt(np.sum(Df2**2))

print("="*60)
print(f"{'Function':<20} {'||Df||_1':<15} {'||Df||_2':<15}")
print("-"*60)
print(f"{'f1 (Step)':<20} {l1_f1:<15.6f} {l2_f1:<15.6f}")
print(f"{'f2 (Smooth)':<20} {l1_f2:<15.6f} {l2_f2:<15.6f}")
print("="*60)
print(f"\nL1 ratio: {l1_f1/l1_f2:.4f}")
print(f"L2 ratio: {l2_f1/l2_f2:.4f}")

## Visualize Functions and Derivatives

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(x, f1, 'bo-', linewidth=2, markersize=8)
axes[0, 0].set_title('Step Function')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].bar(range(n_curve), Df1, color='blue', alpha=0.7, edgecolor='black')
axes[0, 1].set_title('Derivative of Step')
axes[0, 1].grid(True, alpha=0.3, axis='y')

axes[1, 0].plot(x, f2, 'ro-', linewidth=2, markersize=8)
axes[1, 0].set_title('Smooth Sigmoid')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].bar(range(n_curve), Df2, color='red', alpha=0.7, edgecolor='black')
axes[1, 1].set_title('Derivative of Sigmoid')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../results/figures/curve_fitting.png', dpi=150)
plt.show()

## Analysis

L2 regularization prefers f2 (smooth) - small derivatives squared are tiny.
L1 regularization prefers f1 (step) - sparse derivatives (mostly zeros).

# Q15-Q16: IRLS and Total Variation

## Q15: Small Bag - Tikhonov vs Total Variation

In [ ]:
lam_best = 1e-5

print("Solving with Tikhonov...")
result_tikh = cgls(A_small, y_small, L_3d, lam=lam_best, tol=1e-6, max_iter=300)
X_tikh = vector_to_grid(result_tikh.x, (n, n, n))

print(f"Converged: {result_tikh.converged}, Iterations: {result_tikh.iterations}")

## Solve with IRLS

In [ ]:
alpha_best = 0.5

print(f"Solving with IRLS (α={alpha_best})...")
result_tv = irls(
    A_small, y_small, L_3d,
    alpha=alpha_best,
    x0=result_tikh.x,
    epsilon=1e-8,
    tol=1e-4,
    max_outer_iter=20,
    max_inner_iter=100,
    verbose=True
)

X_tv = vector_to_grid(result_tv.x, (n, n, n))

## Compare Reconstructions

In [ ]:
slice_idx = n // 2

compare_reconstructions(
    X_tikh, X_tv,
    labels=("Tikhonov", "Total Variation"),
    slice_idx=slice_idx,
    axis='z',
    save_path='../results/figures/small_comparison.png'
)

## Compare Histograms

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(X_tikh.flatten(), bins=50, edgecolor='black', alpha=0.7)
ax1.set_title('Tikhonov')
ax1.set_xlabel('Density')
ax1.grid(True, alpha=0.3)

ax2.hist(X_tv.flatten(), bins=50, edgecolor='black', alpha=0.7, color='orange')
ax2.set_title('Total Variation')
ax2.set_xlabel('Density')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/small_histograms.png', dpi=150)
plt.show()

## Q16: Large Bag Reconstruction

In [ ]:
y_large, A_large = load_3d_data('../data/Large')
n_large = 49
L_large = build_combined_derivative_matrix(n_large, n_large, dimensions=3)

print(f"Volume: {n_large}^3 = {n_large**3:,} voxels")

## Initialize with Tikhonov

In [ ]:
print("Initializing with Tikhonov...")
result_large_tikh = cgls(
    A_large, y_large, L_large,
    lam=1e-5,
    tol=1e-6,
    max_iter=300
)

print(f"Converged: {result_large_tikh.converged}, Iterations: {result_large_tikh.iterations}")

## Solve with IRLS

In [ ]:
print("Solving with IRLS...")
result_large_tv = irls(
    A_large, y_large, L_large,
    alpha=0.5,
    x0=result_large_tikh.x,
    epsilon=1e-8,
    tol=1e-4,
    max_outer_iter=20,
    max_inner_iter=200,
    verbose=True
)

X_large = vector_to_grid(result_large_tv.x, (n_large, n_large, n_large))

save_reconstruction(
    result_large_tv.x,
    '../results/reconstructions/large_bag_tv.npz',
    metadata={'alpha': 0.5, 'n': n_large, 'converged': result_large_tv.converged}
)

## Visualize Large Bag

In [ ]:
visualize_3d_slices(
    X_large,
    axis='z',
    num_slices=12,
    title="Large Bag Contents (z-slices)",
    save_path='../results/figures/large_z_slices.png'
)

visualize_3d_slices(
    X_large,
    axis='x',
    num_slices=12,
    title="Large Bag Contents (x-slices)",
    save_path='../results/figures/large_x_slices.png'
)

## Analyze Contents

In [ ]:
print("Density Statistics:")
print(f"  Min: {X_large.min():.6f}")
print(f"  Max: {X_large.max():.6f}")
print(f"  Mean: {X_large.mean():.6f}")
print(f"  Std: {X_large.std():.6f}")

plot_histogram(
    X_large.flatten(),
    bins=100,
    title="Large Bag Density Distribution",
    save_path='../results/figures/large_histogram.png'
)

## Analysis

Describe objects found in the bag based on density distribution and spatial location.